# Ring / O-ring / rust inspection: classifier + task-driven RL view selection (Colab)

**Part:** grooved hex-end cylinder. **ACCEPT** = both brown wear rings present + black O-ring present + no rust.
**REJECT** = any ring missing, O-ring missing, or rust.

**Run after (Jetson):** `collect_views.py ... --condition ...` for all placements, then
`build_view_dataset.py`, `build_task_dataset.py`, `sync.py pack-for-colab`.

**Upload:** `mycobot_framework_code.zip` and the newest `to_colab_<stamp>.zip`.

**What happens**
1. A vision transformer (DINOv2 ViT-S/14: self-supervised pre-training, frozen, with a linear head) judges each item
   (ring at hex end, ring at open end, O-ring, rust) from every recorded image; a small CNN is trained the same way as the
   baseline. Both use 5-fold cross-validation **grouped by placement** (a placement is never in its own training fold).
2. The out-of-fold probabilities turn every recorded view into *task evidence* (how surely that image decides each item).
3. The transformer policy is trained by REINFORCE on that evidence per second of safe robot motion; MLP and a
   quality-only policy are the comparisons; `--no-tilt` is the ablation that shows what tilted views add.
4. All planners are evaluated on held-out placements: accuracy of the part verdict vs time budget.

**Download at the end:** `to_jetson_<stamp>.zip` (classifier + policies), `results_task.zip`, `paper_figs.zip`.

In [ ]:
# 1) Upload the two zips
import glob, os
if len(glob.glob('/content/*.zip')) < 2:
    from google.colab import files
    files.upload()
print([os.path.basename(f) for f in glob.glob('/content/*.zip')])

In [ ]:
# 2) Unpack and check
import glob, os, zipfile, json, sys, numpy as np
zips = sorted(glob.glob('/content/*.zip'))
data_zips = [z for z in zips if os.path.basename(z).startswith('to_colab')]
fw_zips = [z for z in zips if not os.path.basename(z).startswith(('to_colab', 'to_jetson', 'results_', 'paper_figs'))]
assert data_zips and fw_zips, 'upload mycobot_framework_code.zip AND to_colab_<stamp>.zip'
for z in fw_zips:
    zipfile.ZipFile(z).extractall('/content/fw')
ROOT = os.path.dirname(sorted(glob.glob('/content/fw/**/task_colab.py', recursive=True), key=len)[0])
os.chdir(ROOT); sys.path.insert(0, ROOT)
zipfile.ZipFile(data_zips[-1]).extractall(ROOT)
for f in ['task_model.py', 'task_colab.py', 'train_views.py', 'views.py', 'geo_attn.py', 'sync.py',
          'data/view_library.npz', 'data/task_crops.npz']:
    print('  %-24s %s' % (f, 'ok' if os.path.exists(f) else 'MISSING'))
assert os.path.exists('data/task_crops.npz'), 'run  python3 build_task_dataset.py  on the Jetson, then pack-for-colab'
import task_model as TM
c = np.load('data/task_crops.npz')
conds = [str(x) for x in c['conditions']]; poses = [str(x) for x in c['poses']]
from collections import Counter
print('\nplacements per pose / condition:')
for k, n in sorted(Counter(zip(poses, conds)).items()): print('  %-10s %-24s %d' % (k[0], k[1], n))
print('crops:', len(c['X']), '| learnable items:', [TM.ITEMS[j] for j in c['learnable']])
miss = [TM.ITEMS[j] for j in range(len(TM.ITEMS)) if j not in set(int(v) for v in c['learnable'])]
if miss: print('WARNING: no good AND defective examples for', miss, '-> these items cannot be learned; record them')

In [ ]:
# 3) Helper
import subprocess, time, torch
def run(args, quiet=False):
    t = time.time(); print('$ python', ' '.join(args), flush=True)
    p = subprocess.Popen([sys.executable] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if not quiet or not line.startswith('iter') or line.startswith('iter     0') or line.startswith('iter  2000'):
            print(line, end='')
    p.wait()
    if p.returncode: print(''.join(lines[-30:])); raise RuntimeError('%s failed' % args[0])
    print('-> %.1f min\n' % ((time.time() - t) / 60))
print('torch', torch.__version__, '| GPU' if torch.cuda.is_available() else '| CPU')

In [ ]:
# 4) Part-state classifiers, both with 5-fold CV grouped by placement:
#    (a) small CNN baseline (not deployed)  (b) DINOv2 ViT-S/14 transformer + linear head (deployed to the Jetson)
run(['task_colab.py', 'classify', '--backbone', 'cnn', '--suffix', '_cnn', '--folds', '5', '--epochs', '30'])
run(['task_colab.py', 'classify', '--backbone', 'vit', '--folds', '5'])
import pandas as pd
cmp = {}
for nm, f in [('CNN', 'results_task/classifier_cv_cnn.json'), ('ViT (DINOv2)', 'results_task/classifier_cv.json')]:
    R = json.load(open(f))
    cmp[nm] = {it: R[it]['auc_all'] for it in R['items']} | {'part verdict acc (all images)': R['part_verdict_all_images']['accuracy'],
               'NumPy ms/crop (Colab CPU)': R.get('numpy_ms_per_crop_colab_cpu'), 'parity': R['numpy_parity_max_abs']}
T = pd.DataFrame(cmp).round(3); display(T); os.makedirs('results_task', exist_ok=True); T.to_csv('results_task/classifier_comparison.csv')
from IPython.display import Image, display
for f in ['results_task/crop_examples.jpg'] + sorted(glob.glob('results_task/*classif*.png')):
    if os.path.exists(f): print(f); display(Image(f, width=700))

In [ ]:
# 5) Task evidence of every recorded view -> RL environment + evidence prior for the Jetson
run(['task_colab.py', 'library'])

In [ ]:
# 6) Smoke test (2 min)
run(['train_views.py', '--data', 'data/view_library_task.npz', '--arch', 'attn', '--seed', '99', '--iters', '50', '--eval-every', '25'])

In [ ]:
# 7) Training: transformer vs MLP on the TASK reward (3 seeds), quality-only policy, and the no-tilt ablation
ITERS = 2000
for arch in ['attn', 'mlp']:
    for s in [0, 1, 2]:
        run(['train_views.py', '--data', 'data/view_library_task.npz', '--arch', arch, '--seed', str(s), '--iters', str(ITERS)], quiet=True)
run(['train_views.py', '--arch', 'attn', '--seed', '0', '--iters', str(ITERS)], quiet=True)      # quality reward (old objective)
try:
    for s in [0, 1, 2]:
        run(['train_views.py', '--data', 'data/view_library_task.npz', '--arch', 'attn', '--seed', str(s),
             '--iters', str(ITERS), '--no-tilt'], quiet=True)
except RuntimeError as e:
    print('no-tilt ablation skipped:', e)

In [ ]:
# 8) Deploy the seed with the best VALIDATION score (never chosen on test), then evaluate on held-out placements
import shutil
def best(pattern):
    c = [f for f in glob.glob(pattern) if '_s99' not in f]
    return max(c, key=lambda f: float(np.load(f)['info_val_pct'])) if c else None
picks = {'attn': best('deploy/view_policy_attn_s[0-9]_task.npz'), 'mlp': best('deploy/view_policy_mlp_s[0-9]_task.npz'),
         'notilt': best('deploy/view_policy_attn_s[0-9]_notilt_task.npz')}
print(picks)
shutil.copy(picks['attn'], 'deploy/view_policy_attn_task.npz'); shutil.copy(picks['mlp'], 'deploy/view_policy_mlp_task.npz')
pol = ['policy_task=deploy/view_policy_attn_task.npz', 'policy_task_mlp=deploy/view_policy_mlp_task.npz',
       'policy_quality=deploy/view_policy_attn_s0.npz']
if picks['notilt']: pol.append('policy_task_notilt=' + picks['notilt'])
run(['task_colab.py', 'evaluate', '--policies'] + pol)
S = json.load(open('results_task/task_eval.json')); print(json.dumps(S, indent=1)[:4000])

In [ ]:
# 9) Seed statistics (held-out placements) and figures
import pandas as pd
rows = []
for f in sorted(glob.glob('results_views/train_*_task/test_summary.json')):
    S = json.load(open(f)); tag = os.path.basename(os.path.dirname(f))[6:]
    if '_s99' in tag: continue
    r = dict(run=tag, policy_pct_of_oracle=S['policy_pct_of_oracle'])
    for c_ in S['comparisons']: r[c_['method'] + '_pct'] = c_['pct_of_oracle']; r['p_vs_' + c_['method']] = c_['wilcoxon_p']
    rows.append(r)
R = pd.DataFrame(rows); R['variant'] = R['run'].str.replace(r'_s\d', '', regex=True)
display(R.groupby('variant').agg(['mean', 'std']).round(2))
for f in sorted(glob.glob('results_task/*.png')) + sorted(glob.glob('paper/figs/*task*.png')):
    print(f); display(Image(f, width=650))

In [ ]:
# 10) Pack for the Jetson and download
run(['sync.py', 'pack-for-jetson'])
shutil.make_archive('/content/results_task', 'zip', ROOT, 'results_task')
shutil.make_archive('/content/results_views', 'zip', ROOT, 'results_views')
os.makedirs('paper/figs', exist_ok=True); shutil.make_archive('/content/paper_figs', 'zip', ROOT, 'paper/figs')
from google.colab import files
files.download(sorted(glob.glob(os.path.join(ROOT, 'to_jetson_*.zip')))[-1])
for z in ['results_task', 'results_views', 'paper_figs']: files.download('/content/%s.zip' % z)
print('Jetson:  python3 sync.py unpack to_jetson_<stamp>.zip  then  run_view_policy.py --task ...')